# HateMirage @ ICON 2026 - 2xT4 QLoRA (dual-GPU, HF-sync, resume, auto-eval)

**Tasks:** A=Target span - B=Intent+Implication generation - **Metrics:** SBERT-cosine + ROUGE-L F1 (greedy T=0)  
**Setup:** Kaggle accelerator `GPU T4 x2` - Internet ON - Secrets: `HF_TOKEN` (write) - Optional: `WANDB_API_KEY`  
**Flow:** env-check -> install -> HF login+config -> data -> RAG -> torchrun-DDP train (both T4s) -> Hub push -> inference -> eval -> Codabench zips.  
If the session hits the 9h limit or converges early, training saves + pushes and the notebook jumps straight to eval - never lose a run. Resume anytime from `HF_REPO`.

Paper: https://arxiv.org/abs/2603.02684 - Task: https://sai-kartheek-reddy.github.io/HateMirage-ICON2026/ - Codabench: https://www.codabench.org/competitions/17783/

In [ ]:
# CELL 1 - env check: must see 2x Tesla T4
import torch, os, time, shutil
T0 = time.time()
print('torch', torch.__version__, '| cuda?', torch.cuda.is_available())
NGPU = torch.cuda.device_count()
print('GPUs:', NGPU)
for i in range(NGPU):
    print(i, torch.cuda.get_device_name(i), f"{torch.cuda.get_device_properties(i).total_memory/1e9:.1f}GB")
!nvidia-smi -L
!df -h /kaggle/working | tail -1
assert NGPU >= 1, 'Enable GPU T4 x2 under Settings > Accelerator'
DUAL = NGPU >= 2
print('DUAL-GPU mode:', DUAL)
OUT = '/kaggle/working/hatemirage_out'
os.makedirs(OUT, exist_ok=True)
os.environ["OUT"] = OUT
print('OUT:', OUT)

In [ ]:
# CELL 2 - deps
!pip install -q -U "transformers>=4.44" accelerate peft trl bitsandbytes "datasets>=2.20" sentence-transformers rouge-score faiss-cpu huggingface_hub pandas scikit-learn openpyxl
import transformers, peft, trl, bitsandbytes
print('transformers', transformers.__version__)

In [ ]:
# CELL 3 - secrets + run config (HF_TOKEN_HARDCODED redacted -> hf_YOUR_TOKEN_HERE in repo copy)
import os
HF_TOKEN_HARDCODED = "hf_YOUR_TOKEN_HERE"
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    HF_TOKEN = HF_TOKEN_HARDCODED
os.environ['HF_TOKEN'] = HF_TOKEN
HAS_HF = bool(HF_TOKEN)
HF_USER = 'blackpanter_252'
MODEL_ID = 'microsoft/Phi-3-mini-128k-instruct'
SMOKE_MODEL = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
EFFECTIVE_MODEL = MODEL_ID if HAS_HF else SMOKE_MODEL
HF_REPO = f'{HF_USER}/hatemirage-phi3-qlora'
WITH_RAG, MAX_RUNTIME_H = True, 8.2
EPOCHS, BS, GRAD_ACCUM, LORA_R = (3, 2, 8, 64) if HAS_HF else (1, 2, 2, 8)


In [ ]:
# CELL 4 - official task data (Development + Evaluation phases)
import glob, pandas as pd, os, urllib.request
GH = 'https://raw.githubusercontent.com/Sai-Kartheek-Reddy/HateMirage-ICON2026/main/'


In [ ]:
# CELL 5 - RAG index (FAISS, all-mpnet-base-v2, top-5)


In [ ]:
# CELL 6a - training config (DDP torchrun)
import os, json, glob


In [ ]:
%%writefile src/train_ddp.py
# часть intentionally shortened - full trainer in src/train_ddp.py in this repo


In [ ]:
%%bash
torchrun --nproc_per_node=2 src/train_ddp.py --config $OUT/train_config.json


In [ ]:
# CELL 7 - inference load (balanced shard, local-first adapter)


In [ ]:
# CELL 8 - inference (v2 official prompt, greedy, OOM-safe) + submissions


In [ ]:
# CELL 9 - eval (SBERT + ROUGE-L) + plots


In [ ]:
# CELL 10 - official submissions (xlsx + taskA/taskB csvs + zips)


In [ ]:
# CELL 11 - resume playbook
